# Healthcare Prior Authorization Q&A  RAG Pipeline

Been thinking about this problem for a while. Prior auth is genuinely one of the most painful parts of the revenue cycle. Providers spend hours on the phone or faxing documents just to get approvals that should be automated.

The idea here: build a RAG system over public CMS coverage guidelines so that a clinician or admin can just *ask* whether a procedure is likely to be covered, and get a grounded answer with source citations.

Data source: CMS Local Coverage Determinations (LCDs) these are publicly available policy documents that define what Medicare covers and under what conditions.

Stack:
- `langchain` for the RAG pipeline
- `sentence-transformers` for embeddings
- `faiss` for vector store
- `transformers` for the LLM layer

In [14]:
#!pip install langchain langchain-community openai sentence-transformers faiss-cpu pandas numpy jupyter

In [16]:
import os
import re
import json
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

## 1. Data CMS Coverage Guidelines

Using CMS's public LCD (Local Coverage Determination) documents. These define the medical necessity criteria for procedures under Medicare Part B.

For this notebook I'm working with a subset I downloaded manually from cms.gov/medicare-coverage-database. Full pipeline would automate this fetch.

In [17]:
# Fetch CMS ICD-10 procedure codes + descriptions
# This is the public CMS data no auth needed
import urllib.request

CMS_URL = "https://www.cms.gov/files/zip/2024-icd-10-cm-codes.zip"

# for now using a local sample  will wire up the full fetch later
# the structure below mirrors what you'd get from the real download

sample_guidelines = [
    {
        "policy_id": "LCD-L33393",
        "title": "Continuous Positive Airway Pressure (CPAP) Therapy",
        "category": "Durable Medical Equipment",
        "coverage_text": """
        CPAP therapy is covered for patients with obstructive sleep apnea (OSA) when:
        1. The diagnosis is established by a complete polysomnogram (PSG) conducted in a facility-based sleep study lab.
        2. The AHI (Apnea-Hypopnea Index) or RDI (Respiratory Disturbance Index) is >= 15 events per hour.
        3. OR: AHI >= 5 and <= 14 with documented symptoms (excessive daytime sleepiness, impaired cognition,
           mood disorders, insomnia, hypertension, ischemic heart disease, or history of stroke).
        Initial coverage is for a 12-week trial period. Continued coverage requires documented adherence
        (usage >= 4 hours per night on >= 70% of nights in a consecutive 30-day period).
        """,
        "icd10_codes": ["G47.33", "G47.30", "G47.31"],
        "effective_date": "2023-01-01"
    },
    {
        "policy_id": "LCD-L34091",
        "title": "Lumbar Spine MRI",
        "category": "Radiology",
        "coverage_text": """
        MRI of the lumbar spine is covered when there is clinical evidence of:
        1. Radiculopathy with neurological signs (motor deficit, reflex changes, positive straight leg raise).
        2. Suspected spinal stenosis with symptoms of neurogenic claudication.
        3. Red flag symptoms: unexplained weight loss, fever, history of cancer, recent trauma,
           progressive neurological deficit, or saddle anesthesia.
        4. Failed conservative therapy >= 6 weeks for non-specific low back pain.
        NOTE: MRI is generally NOT covered for acute low back pain (<6 weeks) without red flag symptoms.
        """,
        "icd10_codes": ["M54.4", "M54.5", "M48.06", "M51.16"],
        "effective_date": "2023-01-01"
    },
    {
        "policy_id": "LCD-L35050",
        "title": "Cardiac Rehabilitation",
        "category": "Rehabilitation Services",
        "coverage_text": """
        Cardiac rehabilitation is covered for patients with:
        1. Acute myocardial infarction within the preceding 12 months.
        2. Coronary artery bypass surgery.
        3. Current stable angina pectoris.
        4. Heart valve repair or replacement.
        5. Percutaneous transluminal coronary angioplasty (PTCA) or coronary stenting.
        6. Heart or heart-lung transplant.
        7. Stable, chronic heart failure (LVEF <= 35%, NYHA class II-IV).
        Coverage: up to 36 sessions over 36 weeks (can extend to 72 sessions with additional documentation).
        """,
        "icd10_codes": ["I21.0", "I21.1", "I21.2", "I20.8", "Z95.1", "I50.22"],
        "effective_date": "2023-01-01"
    },
    {
        "policy_id": "LCD-L36158",
        "title": "Botulinum Toxin Injections",
        "category": "Injections",
        "coverage_text": """
        Botulinum toxin type A (Botox, Dysport) or type B (Myobloc) injections are covered for:
        1. Cervical dystonia (spasmodic torticollis) — ICD-10: G24.3
        2. Blepharospasm — ICD-10: G24.5
        3. Hemifacial spasm — ICD-10: G51.3
        4. Upper limb spasticity following stroke — ICD-10: G83.2, I69.x
        5. Chronic migraine (>= 15 headache days/month) — ICD-10: G43.709
        6. Overactive bladder — when anticholinergic therapy is inadequate — ICD-10: N32.81
        NOT covered for cosmetic purposes. Documentation must include failed response to first-line treatments.
        """,
        "icd10_codes": ["G24.3", "G24.5", "G51.3", "G43.709", "N32.81"],
        "effective_date": "2023-01-01"
    },
    {
        "policy_id": "LCD-L34028",
        "title": "Physical Therapy — General Guidelines",
        "category": "Rehabilitation Services",
        "coverage_text": """
        Physical therapy (PT) is covered when:
        1. The patient has a condition that requires skilled PT services.
        2. Services are provided under a written plan of care certified by a physician.
        3. The patient's condition is expected to improve in a reasonable and predictable timeframe.
        4. Services cannot be safely/effectively performed by the patient or caregiver alone.
        Annual therapy cap: $2,230 (2024) for PT and speech-language pathology combined.
        Exceptions available with KX modifier for medically necessary services above the cap.
        Documentation must include: objective functional goals, measurable progress notes,
        and justification for continued skilled care at each visit.
        """,
        "icd10_codes": ["M54.5", "M79.3", "S72.001A", "G35", "M16.11"],
        "effective_date": "2024-01-01"
    }
]

df = pd.DataFrame(sample_guidelines)
print(f"loaded {len(df)} coverage policies")
df[['policy_id', 'title', 'category']].head(10)

loaded 5 coverage policies


,policy_id,title,category
0,LCD-L33393,Continuous Positive Airway Pressure (CPAP) The...,Durable Medical Equipment
1,LCD-L34091,Lumbar Spine MRI,Radiology
2,LCD-L35050,Cardiac Rehabilitation,Rehabilitation Services
3,LCD-L36158,Botulinum Toxin Injections,Injections
4,LCD-L34028,Physical Therapy — General Guidelines,Rehabilitation Services


## 2. Text Chunking

Before embedding, need to chunk the coverage text. The tricky part here is that these policy docs have specific structure (numbered criteria, exceptions, coverage limits). I want to preserve that context rather than split mid sentence.

Going with a simple recursive splitter first, then I'll see if the retrieval quality is good enough.

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def build_documents(guidelines_df):
    docs = []
    for _, row in guidelines_df.iterrows():
        # include metadata — this gets surfaced in citations
        doc = Document(
            page_content=row['coverage_text'].strip(),
            metadata={
                'policy_id': row['policy_id'],
                'title': row['title'],
                'category': row['category'],
                'icd10_codes': ', '.join(row['icd10_codes']),
                'effective_date': row['effective_date']
            }
        )
        docs.append(doc)
    return docs

raw_docs = build_documents(df)

# chunk size tuned for policy docs  these have dense criteria lists
# smaller chunks = more precise retrieval but lose surrounding context
# 500/100 overlap felt right after a few test queries
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

chunked_docs = splitter.split_documents(raw_docs)
print(f"split {len(raw_docs)} docs into {len(chunked_docs)} chunks")

# sanity check look at one chunk
print("\n--- sample chunk ---")
print(chunked_docs[2].page_content)
print("\nmetadata:", chunked_docs[2].metadata)

split 5 docs into 10 chunks

--- sample chunk ---
MRI of the lumbar spine is covered when there is clinical evidence of:
        1. Radiculopathy with neurological signs (motor deficit, reflex changes, positive straight leg raise).
        2. Suspected spinal stenosis with symptoms of neurogenic claudication.
        3. Red flag symptoms: unexplained weight loss, fever, history of cancer, recent trauma, 
           progressive neurological deficit, or saddle anesthesia.

metadata: {'policy_id': 'LCD-L34091', 'title': 'Lumbar Spine MRI', 'category': 'Radiology', 'icd10_codes': 'M54.4, M54.5, M48.06, M51.16', 'effective_date': '2023-01-01'}


## 3. Embeddings + Vector Store

Using `sentence-transformers/all-MiniLM-L6-v2` it's fast, runs locally, and works well for semantic similarity on clinical text.

Tried `all-mpnet-base-v2` too but the latency wasn't worth it for this use case. If this were going to production I'd evaluate a healthcare-specific model like `BioBERT` or `ClinicalBERT`.

FAISS for the vector store simple, fast, no infra needed for prototyping.

In [19]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

print("building vector store...")
vectorstore = FAISS.from_documents(chunked_docs, embeddings)

# save it so we don't have to rebuild every time
vectorstore.save_local(str(DATA_DIR / 'faiss_cms_index'))
print("done. index saved to data/faiss_cms_index")

loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


building vector store...
done. index saved to data/faiss_cms_index


In [20]:
# check vectorstore loaded correctly
print(type(vectorstore))
print(vectorstore.index.ntotal)  # should print a number > 0

<class 'langchain_community.vectorstores.faiss.FAISS'>
10


In [21]:
# quick retrieval test before wiring up the LLM
# want to make sure the right chunks come back for a sample query

test_query = "Is an MRI covered for back pain?"
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke(test_query)
print(f"retrieved {len(results)} chunks for: '{test_query}'\n")
for i, r in enumerate(results):
    print(f"--- result {i+1} ({r.metadata['policy_id']}: {r.metadata['title']}) ---")
    print(r.page_content[:300])
    print()

retrieved 3 chunks for: 'Is an MRI covered for back pain?'

--- result 1 (LCD-L34091: Lumbar Spine MRI) ---
MRI of the lumbar spine is covered when there is clinical evidence of:
        1. Radiculopathy with neurological signs (motor deficit, reflex changes, positive straight leg raise).
        2. Suspected spinal stenosis with symptoms of neurogenic claudication.
        3. Red flag symptoms: unexplain

--- result 2 (LCD-L34091: Lumbar Spine MRI) ---
progressive neurological deficit, or saddle anesthesia.
        4. Failed conservative therapy >= 6 weeks for non-specific low back pain.
        NOTE: MRI is generally NOT covered for acute low back pain (<6 weeks) without red flag symptoms.

--- result 3 (LCD-L34028: Physical Therapy — General Guidelines) ---
Physical therapy (PT) is covered when:
        1. The patient has a condition that requires skilled PT services.
        2. Services are provided under a written plan of care certified by a physician.
        3. The patient's con

## 4. RAG Pipeline

Now the actual Q&A chain. Using `RetrievalQA` with a custom prompt that:
1. Grounds answers strictly in retrieved context (no hallucination)
2. Always surfaces the source policy ID for citations
3. Flags when something is NOT covered — that's actually the most useful output for prior auth workflows

LLM: defaulting to GROQ

In [22]:
!pip install -q langchain


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 11.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, which is not installed.


In [23]:
!pip show langchain | grep Version

Version: 1.3.2


In [24]:
!pip uninstall -y langchain langchain-core langchain-community langchain-groq langchain-google-genai langchain-text-splitters 2>/dev/null
!pip install -q "langchain-core==0.3.28" "langchain-community==0.3.14" "langchain-text-splitters==0.3.4" "langchain-groq==0.2.3" sentence-transformers faiss-cpu

Found existing installation: langchain 1.3.2
Uninstalling langchain-1.3.2:
  Successfully uninstalled langchain-1.3.2
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
ERROR: Cannot install langchain-community==0.3.14 and langchain-core==0.3.28 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [29]:
import requests

GROQ_API_KEY = "<GROK API KEY>"

response = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "llama-3.1-8b-instant",
        "messages": [{"role": "user", "content": "say hello"}],
        "temperature": 0
    },
    timeout=30
)

print("status code:", response.status_code)
print("response:", response.text)

status code: 200
response: {"id":"chatcmpl-fe145f0a-13c9-4e12-8673-ed30b396b25e","object":"chat.completion","created":1780354324,"model":"llama-3.1-8b-instant","choices":[{"index":0,"message":{"role":"assistant","content":"Hello. How can I assist you today?"},"logprobs":null,"finish_reason":"stop"}],"usage":{"queue_time":0.168945388,"prompt_tokens":37,"prompt_time":0.002564329,"completion_tokens":10,"completion_time":0.010316192,"total_tokens":47,"total_time":0.012880521},"usage_breakdown":null,"system_fingerprint":"fp_7ccc667439","x_groq":{"id":"req_01kt2p65fmejrvmeh69ywzgr8r","seed":664488196},"service_tier":"on_demand"}



In [30]:
import requests

GROQ_API_KEY = "<GROK API KEY>"

def ask(question):
    # get relevant docs from vectorstore
    docs = vectorstore.similarity_search(question, k=3)
    context = "\n\n".join([
        f"[{d.metadata['policy_id']}] {d.page_content}"
        for d in docs
    ])

    prompt = f"""You are a healthcare prior authorization specialist with expertise in CMS Medicare coverage policies.

Use ONLY the coverage guidelines provided below to answer the question.
Do not use any outside knowledge. If the answer is not in the guidelines, say so explicitly.

When answering:
- State clearly whether the procedure/service is COVERED, NOT COVERED, or CONDITIONALLY COVERED
- List the specific criteria that must be met
- Cite the policy ID (e.g., LCD-L33393)
- Flag any documentation requirements
- Note any important exclusions

Coverage Guidelines:
{context}

Question: {question}

Answer:"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
    )

    result = response.json()
    print(f"Q: {question}")
    print("=" * 60)
    print(result)
    print(result["choices"][0]["message"]["content"])
    print()

print("chain ready")

chain ready


## 5. Test Queries

Running through a few realistic prior auth scenarios. These mirror the kinds of questions a billing team would actually ask.

In [31]:
# test 1
ask("My patient has had back pain for 3 weeks with no other symptoms. Is an MRI covered?")


Q: My patient has had back pain for 3 weeks with no other symptoms. Is an MRI covered?
{'id': 'chatcmpl-c92ba8e1-9ab9-4616-a123-05cd19ffed5d', 'object': 'chat.completion', 'created': 1780354335, 'model': 'llama-3.1-8b-instant', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Based on the provided coverage guidelines, the answer is:\n\nNOT COVERED\n\nSpecific criteria that must be met for MRI coverage:\n- Clinical evidence of one of the following:\n  1. Radiculopathy with neurological signs (motor deficit, reflex changes, positive straight leg raise).\n  2. Suspected spinal stenosis with symptoms of neurogenic claudication.\n  3. Red flag symptoms: unexplained weight loss, fever, history of cancer, recent trauma, \n     progressive neurological deficit, or saddle anesthesia.\n\nSince the patient has no red flag symptoms and has only had back pain for 3 weeks, the MRI is not covered.\n\nPolicy ID: LCD-L34091\nFlagged documentation requirement: None explicitly stated

In [32]:
# test 2
ask("Patient has AHI of 10 with excessive daytime sleepiness. Will CPAP be covered?")

Q: Patient has AHI of 10 with excessive daytime sleepiness. Will CPAP be covered?
{'id': 'chatcmpl-1f869147-6850-4ac8-8cc2-2d8077a5a513', 'object': 'chat.completion', 'created': 1780354348, 'model': 'llama-3.1-8b-instant', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'COVERED\n\nSpecific criteria that must be met:\n1. AHI >= 5 (which is met with AHI of 10)\n2. Documented symptoms (excessive daytime sleepiness, which is present in this case)\n3. Diagnosis is not required to be established by a complete polysomnogram (PSG) in this scenario, as the AHI is >= 5 and <= 14.\n\nPolicy ID: LCD-L33393\n\nNo documentation requirements flagged, as the initial coverage criteria are met.\n\nImportant exclusion: None mentioned in the provided guidelines.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'queue_time': 0.344565015, 'prompt_tokens': 404, 'prompt_time': 0.310569678, 'completion_tokens': 116, 'completion_time': 0.226499753, 'total_tokens': 520, 'total_time

In [33]:
# test 3
ask("How many physical therapy sessions does Medicare cover per year?")

Q: How many physical therapy sessions does Medicare cover per year?
{'id': 'chatcmpl-803c7a77-7824-435e-bd64-50d3b24ca2f4', 'object': 'chat.completion', 'created': 1780354350, 'model': 'llama-3.1-8b-instant', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Based on the provided coverage guidelines, the answer is:\n\nCOVERED\n- Up to $2,230 (2024) for physical therapy (PT) and speech-language pathology combined per year.\n- This is specified in LCD-L34028.\n\nImportant note: This coverage limit applies to both physical therapy and speech-language pathology services combined.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'queue_time': 0.063393929, 'prompt_tokens': 457, 'prompt_time': 0.04602573, 'completion_tokens': 68, 'completion_time': 0.125615935, 'total_tokens': 525, 'total_time': 0.171641665}, 'usage_breakdown': None, 'system_fingerprint': 'fp_e2c608b1d6', 'x_groq': {'id': 'req_01kt2p6z9nf0aaxarxsp6bg7r5', 'seed': 948446290}, 'service_tier': 'on_de

In [34]:
# test 4
ask("My patient had a heart attack 8 months ago. Are they eligible for cardiac rehab?")

Q: My patient had a heart attack 8 months ago. Are they eligible for cardiac rehab?
{'id': 'chatcmpl-142172bb-7245-4fa8-8465-cbc69e506054', 'object': 'chat.completion', 'created': 1780354361, 'model': 'llama-3.1-8b-instant', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'COVERED\n\nCriteria that must be met:\n1. The patient had an acute myocardial infarction within the preceding 12 months.\n \nPolicy ID: LCD-L35050\n\nNo documentation requirements flagged.\n\nImportant exclusion: None mentioned in the provided guidelines.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'queue_time': 0.062548149, 'prompt_tokens': 451, 'prompt_time': 0.397630594, 'completion_tokens': 53, 'completion_time': 0.215650963, 'total_tokens': 504, 'total_time': 0.613281557}, 'usage_breakdown': None, 'system_fingerprint': 'fp_020e283281', 'x_groq': {'id': 'req_01kt2p791aeyzv00p8djmsheee', 'seed': 964368415}, 'service_tier': 'on_demand'}
COVERED

Criteria that must be met:
1. The p

## 6. Retrieval Quality Check

Before calling this done, want to sanity check retrieval quality more systematically. The main failure mode I'm worried about: query uses clinical terminology that doesn't match the policy document language.

e.g., a clinician might ask about "obstructive sleep apnea" while the policy says "OSA" the embedding model should handle this but worth verifying.

In [35]:
# test retrieval with clinical synonyms
synonym_tests = [
    ("obstructive sleep apnea CPAP coverage", "LCD-L33393"),
    ("lumbar spine imaging criteria", "LCD-L34091"),
    ("botox injection for migraine", "LCD-L36158"),
    ("PT treatment plan documentation", "LCD-L34028"),
]

print("Retrieval accuracy check:\n")
hits = 0
for query, expected_policy in synonym_tests:
    results = retriever.get_relevant_documents(query)
    retrieved_ids = [r.metadata['policy_id'] for r in results]
    hit = expected_policy in retrieved_ids
    hits += int(hit)
    status = "✓" if hit else "✗"
    print(f"{status} '{query}' -> expected {expected_policy}, got {retrieved_ids}")

print(f"\nretrieval accuracy: {hits}/{len(synonym_tests)} = {hits/len(synonym_tests)*100:.0f}%")

Retrieval accuracy check:

✓ 'obstructive sleep apnea CPAP coverage' -> expected LCD-L33393, got ['LCD-L33393', 'LCD-L33393', 'LCD-L35050']
✓ 'lumbar spine imaging criteria' -> expected LCD-L34091, got ['LCD-L34091', 'LCD-L34091', 'LCD-L35050']
✓ 'botox injection for migraine' -> expected LCD-L36158, got ['LCD-L36158', 'LCD-L34028', 'LCD-L33393']
✓ 'PT treatment plan documentation' -> expected LCD-L34028, got ['LCD-L34028', 'LCD-L34028', 'LCD-L35050']

retrieval accuracy: 4/4 = 100%


## 7. What's Next

This is a working prototype here's what I'd do to take it to production:

**Data:**
- Ingest the full CMS LCD/NCD database (700+ policies) via their public API
- Add payer-specific policies (BCBS, Aetna, UHC) where publicly available
- Auto-refresh when policy effective dates change

**Retrieval:**
- Swap `all-MiniLM-L6-v2` for a healthcare-tuned model (ClinicalBERT, BioBERT)
- Add a re-ranking step retrieve top 10, re-rank to top 3 for LLM context
- Hybrid search: dense embeddings + BM25 keyword matching for ICD codes

**LLM layer:**
- Fine-tune on prior auth approval/denial decisions to learn the "tone" of policy interpretation
- Add structured output: return JSON with {covered: bool, criteria_met: [], documentation_needed: [], policy_citations: []}
- Confidence scoring flag low-confidence answers for human review

**Infrastructure:**
- HIPAA compliance: all data stays on-prem or in a BAA-covered cloud environment
- Audit logging: every query + response logged for compliance
- FastAPI wrapper + simple Streamlit UI for non-technical staff

**Evaluation:**
- Need a labeled test set of real prior auth decisions to measure accuracy
- Track false negative rate (saying "not covered" when it is) this is the costly error

The core insight: this isn't replacing clinical judgment, it's reducing the lookup time from 20 minutes to 20 seconds.

In [38]:
import time
import numpy as np

queries = [
    "Is CPAP covered for mild sleep apnea?",
    "What are the cardiac rehab session limits?",
    "When is botox covered for headaches?"
]

times = []
for q in queries:
    start = time.time()
    ask(q)
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"latency: {elapsed:.2f}s\n")

print(f"avg latency: {np.mean(times):.2f}s")

Q: Is CPAP covered for mild sleep apnea?
{'id': 'chatcmpl-316c230b-bb66-4a92-97ce-f779a958f3e7', 'object': 'chat.completion', 'created': 1780354477, 'model': 'llama-3.1-8b-instant', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Based on the provided coverage guidelines, CPAP therapy is CONDITIONALLY COVERED for patients with obstructive sleep apnea (OSA) when:\n\n1. The diagnosis is established by a complete polysomnogram (PSG) conducted in a facility-based sleep study lab.\n2. The AHI (Apnea-Hypopnea Index) or RDI (Respiratory Disturbance Index) is >= 15 events per hour.\n3. OR: AHI >= 5 and <= 14 with documented symptoms (excessive daytime sleepiness, impaired cognition, mood disorders, insomnia, hypertension, ischemic heart disease, or history of stroke).\n\nSince the question specifies "mild sleep apnea," it implies an AHI of < 15 events per hour. Therefore, CPAP therapy would not be covered based on the AHI criteria alone.\n\nHowever, if the patient has doc